In [45]:
!pip install -q Sastrawi nltk vaderSentiment tqdm seaborn scikit-learn tensorflow gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 24.6 MB/s eta 0:00:00


In [47]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import tensorflow as tf

from tqdm import tqdm
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import nltk
nltk.download('punkt_tab')

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Bidirectional, LSTM)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from gensim.models import Word2Vec

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [48]:
nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [49]:
try:
    _vader_analyzer = SentimentIntensityAnalyzer()
    VADER_AVAILABLE = True
    print("VADER tersedia dan siap digunakan.")
except Exception:
    _vader_analyzer = None
    VADER_AVAILABLE = False
    print("VADER tidak ditemukan — menggunakan leksikon sebagai cadangan.")


VADER tersedia dan siap digunakan.


In [50]:
file_id = "1n1wO7z9ZeUSHLdE8tcnq6f0PpFZx93tA"
drive_url = f"https://drive.google.com/uc?export=download&id={file_id}"
df = pd.read_csv(drive_url)
print("Dataset dimuat — shape:", df.shape)
display(df.head(5))

Dataset dimuat — shape: (10000, 4)


,userName,score,at,content
0,Pengguna Google,1,2025-10-26 14:15:32,"mengecewakan sih, barang sudah 2 hari status s..."
1,Pengguna Google,5,2025-10-26 06:42:29,"bagus bngt, kurir nya pada ramah, semoga bang ..."
2,Pengguna Google,1,2025-10-22 05:09:17,"Dua kali pesan, selalu dtg terlambat. yg skrg ..."
3,Pengguna Google,2,2025-10-19 05:57:05,"waktu saya lacak pesanan saya, tertulis disera..."
4,Pengguna Google,5,2025-10-22 16:07:11,Aplikasi Klik Indomaret sangat membantu untuk ...


In [51]:
TEXT_COL = "content"
LABEL_COL = "sentiment"



In [52]:
def auto_label(text):
    if VADER_AVAILABLE:
        scores = _vader_analyzer.polarity_scores(text or "")
        comp = scores.get("compound", 0.0)
        if comp >= 0.05:
            return "positive"
        if comp <= -0.05:
            return "negative"
        return "neutral"

In [53]:
if LABEL_COL not in df.columns or df[LABEL_COL].isnull().all() or (df[LABEL_COL].astype(str).str.strip() == "").all():
    tqdm.pandas(desc="Pelabelan otomatis")
    df[LABEL_COL] = df[TEXT_COL].progress_map(auto_label)

df[LABEL_COL] = df[LABEL_COL].astype(str).str.lower().map({
    "positive":"positive","pos":"positive","positif":"positive",
    "neutral":"neutral","netral":"neutral",
    "negative":"negative","neg":"negative","negatif":"negative"
}).fillna(df[LABEL_COL])

print("\nDistribusi label setelah pelabelan:")
print(df[LABEL_COL].value_counts())

Pelabelan otomatis: 100%|██████████| 10000/10000 [00:00<00:00, 12379.81it/s]


Distribusi label setelah pelabelan:
sentiment
neutral     7182
positive    1644
negative    1174
Name: count, dtype: int64


In [54]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()
stop_words = set(stopwords.words('indonesian'))

def normalize_text(text):
    s = str(text or "").lower()
    s = re.sub(r"http\S+|www\S+", " ", s)
    s = re.sub(r"@\w+|#", " ", s)
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    toks = word_tokenize(s)
    toks = [t for t in toks if t not in stop_words and len(t) > 1]
    stems = [stemmer.stem(t) for t in toks]
    return " ".join(stems)

tqdm.pandas(desc="Cleaning texts")
df["cleaned"] = df[TEXT_COL].progress_apply(normalize_text)

print("\nContoh hasil cleaning:")
display(df[[TEXT_COL, "cleaned", LABEL_COL]].head(5))

Cleaning texts: 100%|██████████| 10000/10000 [11:53<00:00, 14.01it/s]


Contoh hasil cleaning:


,content,cleaned,sentiment
0,"mengecewakan sih, barang sudah 2 hari status s...",kecewa sih barang status selesai proses kompla...,neutral
1,"bagus bngt, kurir nya pada ramah, semoga bang ...",bagus bngt kurir nya ramah moga bang kurir ram...,positive
2,"Dua kali pesan, selalu dtg terlambat. yg skrg ...",kali pesan dtg lambat yg skrg blm pilih kirim ...,neutral
3,"waktu saya lacak pesanan saya, tertulis disera...",lacak pesan tulis serah grab karna pesan tungg...,neutral
4,Aplikasi Klik Indomaret sangat membantu untuk ...,aplikasi klik indomaret bantu belanja butuh ha...,neutral


In [55]:
label_to_int = {"negative":0, "neutral":1, "positive":2}
int_to_label = {v:k for k,v in label_to_int.items()}
df["label_num"] = df[LABEL_COL].map(label_to_int)
df = df.dropna(subset=["label_num"]).reset_index(drop=True)
df["label_num"] = df["label_num"].astype(int)

In [56]:
MAX_VOCAB = 20000
MAX_SEQ = 200
NUM_CLASSES = 3

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(df["cleaned"])
vocab_used = min(MAX_VOCAB, len(tokenizer.word_index) + 1)

X_all = pad_sequences(tokenizer.texts_to_sequences(df["cleaned"]), maxlen=MAX_SEQ, padding='post')
y_all = df["label_num"].values

def compute_cw(y_arr):
    classes = np.unique(y_arr)
    weights = compute_class_weight("balanced", classes=classes, y=y_arr)
    return dict(zip(classes, weights))

def create_cnn_model(vocab_size, embed_dim=128, seq_len=MAX_SEQ, n_classes=NUM_CLASSES):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=seq_len),
        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def create_bilstm_model(vocab_size, embed_dim=128, seq_len=MAX_SEQ, lstm_units=128, n_classes=NUM_CLASSES):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=seq_len),
        Bidirectional(LSTM(lstm_units, dropout=0.3)),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

es_cb = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
rlr_cb = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)



df[LABEL_COL] = df[LABEL_COL].astype(str).str.lower().map({
    "positive":"positive","pos":"positive","positif":"positive",
    "neutral":"neutral","netral":"neutral",
    "negative":"negative","neg":"negative","negatif":"negative"
}).fillna(df[LABEL_COL])

print("\nDistribusi label setelah pelabelan:")
print(df[LABEL_COL].value_counts())


Distribusi label setelah pelabelan:
sentiment
neutral     7182
positive    1644
negative    1174
Name: count, dtype: int64


In [57]:
factory = StemmerFactory()
_indonesian_stemmer = factory.create_stemmer()
_stopwords = set(stopwords.words('indonesian'))

def normalize_text(text):
    s = str(text or "").lower()
    s = re.sub(r"http\S+|www\S+", " ", s)
    s = re.sub(r"@\w+|#", " ", s)
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    toks = word_tokenize(s)
    toks = [t for t in toks if t not in _stopwords and len(t) > 1]
    stems = [ _indonesian_stemmer.stem(t) for t in toks ]
    return " ".join(stems)


In [61]:
def compute_cw(y_arr):
    classes = np.unique(y_arr)
    weights = compute_class_weight("balanced", classes=classes, y=y_arr)
    return dict(zip(classes, weights))

In [62]:
es_cb = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
rlr_cb = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

In [63]:
X_tr1, X_te1, y_tr1, y_te1 = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=7)
model_cnn = create_cnn_model(vocab_used)
cw1 = compute_cw(y_tr1)
print("Training CNN (80/20) — class weights:", cw1)
model_cnn.fit(X_tr1, y_tr1, validation_data=(X_te1, y_te1),
              epochs=10, batch_size=128, class_weight=cw1,
              callbacks=[es_cb, rlr_cb], verbose=1)
loss_cnn, acc_cnn = model_cnn.evaluate(X_te1, y_te1, verbose=0)

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_all, y_all, test_size=0.3, stratify=y_all, random_state=42)
model_bilstm = create_bilstm_model(vocab_used)
cw2 = compute_cw(y_tr2)
print("Training BiLSTM (70/30) — class weights:", cw2)
model_bilstm.fit(X_tr2, y_tr2, validation_data=(X_te2, y_te2),
                 epochs=10, batch_size=128, class_weight=cw2,
                 callbacks=[es_cb, rlr_cb], verbose=1)
loss_bilstm, acc_bilstm = model_bilstm.evaluate(X_te2, y_te2, verbose=0)

Training CNN (80/20) — class weights: {np.int64(0): np.float64(2.839900603478878), np.int64(1): np.float64(0.4640909618285184), np.int64(2): np.float64(2.0278833967046896)}
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 232ms/step - accuracy: 0.3512 - loss: 1.0836 - val_accuracy: 0.6765 - val_loss: 0.9424 - learning_rate: 0.0010
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 221ms/step - accuracy: 0.7253 - loss: 0.7456 - val_accuracy: 0.8155 - val_loss: 0.5916 - learning_rate: 0.0010
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 224ms/step - accuracy: 0.8988 - loss: 0.3813 - val_accuracy: 0.8805 - val_loss: 0.4231 - learning_rate: 0.0010
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 21s 234ms/step - accuracy: 0.9478 - loss: 0.2140 - val_accuracy: 0.8215 - val_loss: 0.5101 - learning_rate: 0.0010
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step - accuracy: 0.9593 - loss: 0.1448
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 229ms/step - accuracy: 0.9594 - loss: 0.1446 - val_accuracy: 0.8680 - val_loss: 0.4348 - learning_rate: 0.0010
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 222ms/step - accuracy: 0.9817 - loss: 0.0761 - val_ac

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


55/55 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.4059 - loss: 1.0826 - val_accuracy: 0.5380 - val_loss: 0.9372 - learning_rate: 0.0010
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.6313 - loss: 0.7864 - val_accuracy: 0.7430 - val_loss: 0.7020 - learning_rate: 0.0010
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.8117 - loss: 0.5562 - val_accuracy: 0.8237 - val_loss: 0.5331 - learning_rate: 0.0010
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 56s 1s/step - accuracy: 0.8768 - loss: 0.3592 - val_accuracy: 0.8213 - val_loss: 0.5078 - learning_rate: 0.0010
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.9211 - loss: 0.2335 - val_accuracy: 0.8380 - val_loss: 0.5348 - learning_rate: 0.0010
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 887ms/step - accuracy: 0.9429 - loss: 0.1940
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
55/55 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.9429 - loss: 0.1939 - val_accuracy: 0.8400 - v

In [65]:
print("\n===== Percobaan 1: TF-IDF + ML (70/30) =====")
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(
    df["cleaned"], df["label_num"], test_size=0.3,
    stratify=df["label_num"], random_state=42
)

cw_tfidf = compute_class_weight("balanced", classes=np.unique(y_train_tfidf), y=y_train_tfidf)
cw_tfidf = dict(zip(np.unique(y_train_tfidf), cw_tfidf))

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf_vec = tfidf.fit_transform(X_train_tfidf)
X_test_tfidf_vec = tfidf.transform(X_test_tfidf)

models_tfidf = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight=cw_tfidf),
    "SVM": LinearSVC(class_weight=cw_tfidf),
    "Random Forest (TF-IDF)": RandomForestClassifier(n_estimators=200, random_state=42)
}

for name, model in models_tfidf.items():
    model.fit(X_train_tfidf_vec, y_train_tfidf)
    preds = model.predict(X_test_tfidf_vec)
    acc = accuracy_score(y_test_tfidf, preds)
    print(f"\n{name} Accuracy: {acc*100:.2f}%")





===== Percobaan 1: TF-IDF + ML (70/30) =====

Logistic Regression Accuracy: 82.20%

SVM Accuracy: 87.33%

Random Forest (TF-IDF) Accuracy: 88.43%


In [67]:
print("\n===== Percobaan 2: Word2Vec + Random Forest (80/20) =====")
sentences = [text.split() for text in df["cleaned"]]
w2v_model = Word2Vec(sentences, vector_size=100, window=5,
                     min_count=2, workers=4, sg=1)

def vectorize_text(texts, model):
    vectors = []
    for txt in texts:
        words = [w for w in txt.split() if w in model.wv]
        if len(words) > 0:
            vec = np.mean(model.wv[words], axis=0)
        else:
            vec = np.zeros(model.vector_size)
        vectors.append(vec)
    return np.array(vectors)

X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    df["cleaned"], df["label_num"], test_size=0.2,
    stratify=df["label_num"], random_state=7
)

X_train_w2v_vec = vectorize_text(X_train_w2v, w2v_model)
X_test_w2v_vec = vectorize_text(X_test_w2v, w2v_model)

rf_w2v = RandomForestClassifier(n_estimators=300, random_state=42)
rf_w2v.fit(X_train_w2v_vec, y_train_w2v)
pred_w2v = rf_w2v.predict(X_test_w2v_vec)

acc_w2v = accuracy_score(y_test_w2v, pred_w2v)
print(f"\nRandom Forest (Word2Vec) Accuracy: {acc_w2v*100:.2f}%")




===== Percobaan 2: Word2Vec + Random Forest (80/20) =====

Random Forest (Word2Vec) Accuracy: 74.15%


In [70]:
!pip freeze > requirements.txt
from google.colab import files
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>